In [1]:
import sys
print(sys.version)

3.13.0 (tags/v3.13.0:60403a5, Oct  7 2024, 09:38:07) [MSC v.1941 64 bit (AMD64)]


In [2]:
import pandas as pd
import numpy as np
import sklearn
import joblib

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)


Pandas: 2.2.3
NumPy: 2.2.5
Scikit-learn: 1.6.1
Joblib: 1.5.0


# 02 — Feature Engineering & Context Preprocessing

This notebook performs feature selection, feature engineering, skewness analysis,
categorical encoding, numerical normalization, and context vector generation
for the LinUCB recommendation model.

In [5]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles and folders here:")
print(os.listdir("."))

Current working directory:
c:\Users\Bhoomika\OneDrive\Desktop\final year project

Files and folders here:
['02_feature_engineering.ipynb', 'data_dictionary.csv', 'data_quality_report.csv', 'Final_year_project.ipynb', 'test_business_context_cleaned.csv', 'train_business_context_cleaned.csv']


## 2.1 Feature Selection

The cleaned dataset contains business, owner, financial, digital adoption,
and business constraint information. Only features that represent the
business context available for recommendation are retained.

Identifier columns, constant metadata, and outcome-like variables that may
introduce leakage are excluded from the LinUCB context vector.

In [9]:
# Selected context features for the recommendation model

context_features = [
    "city",
    "sector",
    "female_majority_owned",
    "migrant_owner",
    "previously_unemployed_owner",
    "primary_income_earner",
    "owner_has_contract_job",
    "household_size_proxy",
    "employee_count",
    "bank_account",
    "has_loan",
    "friends_family_finance",
    "sells_on_credit",
    "buys_on_credit",
    "mobile_money",
    "computer_or_tablet",
    "electricity",
    "digital_adoption_score",
    "household_based_business",
    "non_fixed_premises",
    "pl_statement",
    "management_score_proxy",
    "monthly_revenue_proxy_inr",
    "monthly_profit_proxy_inr",
    "finance_constraint",
    "customer_reach_constraint",
    "management_constraint"
]

excluded_features = [
    "business_id",
    "country",
    "data_origin",
    "profit_last_month"
]

print("Number of selected features:", len(context_features))
print("\nExcluded features:", excluded_features)

Number of selected features: 27

Excluded features: ['business_id', 'country', 'data_origin', 'profit_last_month']


In [11]:
# Final selected context features

context_features = [
    "city",
    "sector",
    "female_majority_owned",
    "migrant_owner",
    "previously_unemployed_owner",
    "primary_income_earner",
    "owner_has_contract_job",
    "household_size_proxy",
    "employee_count",
    "bank_account",
    "has_loan",
    "friends_family_finance",
    "sells_on_credit",
    "buys_on_credit",
    "mobile_money",
    "computer_or_tablet",
    "electricity",
    "digital_adoption_score",
    "household_based_business",
    "non_fixed_premises",
    "pl_statement",
    "management_score_proxy",
    "monthly_revenue_proxy_inr",
    "monthly_profit_proxy_inr",
    "profit_margin",
    "revenue_per_employee",
    "finance_constraint",
    "customer_reach_constraint",
    "management_constraint"
]

print("Final number of context features:", len(context_features))

Final number of context features: 29


In [12]:
# Check skewness of numerical features

numerical_features = [
    "female_majority_owned",
    "migrant_owner",
    "previously_unemployed_owner",
    "primary_income_earner",
    "owner_has_contract_job",
    "household_size_proxy",
    "employee_count",
    "bank_account",
    "has_loan",
    "friends_family_finance",
    "sells_on_credit",
    "buys_on_credit",
    "mobile_money",
    "computer_or_tablet",
    "electricity",
    "digital_adoption_score",
    "household_based_business",
    "non_fixed_premises",
    "pl_statement",
    "management_score_proxy",
    "monthly_revenue_proxy_inr",
    "monthly_profit_proxy_inr",
    "profit_margin",
    "revenue_per_employee",
    "finance_constraint",
    "customer_reach_constraint",
    "management_constraint"
]

skewness = train_df[numerical_features].skew().sort_values(ascending=False)

print("Skewness of numerical features:")
print(skewness)

Skewness of numerical features:
monthly_profit_proxy_inr       4.685438
computer_or_tablet             2.714182
owner_has_contract_job         2.513667
migrant_owner                  2.302777
has_loan                       2.250540
non_fixed_premises             2.200256
female_majority_owned          2.167750
monthly_revenue_proxy_inr      2.027468
revenue_per_employee           1.866728
household_based_business       1.489162
previously_unemployed_owner    1.350520
friends_family_finance         1.099942
sells_on_credit                1.034214
profit_margin                  0.945442
buys_on_credit                 0.916755
management_score_proxy         0.883197
pl_statement                   0.708439
bank_account                   0.684602
employee_count                 0.638269
digital_adoption_score         0.427569
mobile_money                   0.336080
household_size_proxy          -0.120458
electricity                   -0.135905
management_constraint         -0.453091
primary_

## 2.5 Skewness Investigation and Log Transformation

Skewness was investigated before applying transformations.

The financial features `monthly_profit_proxy_inr` and
`monthly_revenue_proxy_inr` showed substantial right skewness. The derived
feature `revenue_per_employee` was also right-skewed.

Therefore, `np.log1p()` will be applied to:

- `monthly_profit_proxy_inr`
- `monthly_revenue_proxy_inr`
- `revenue_per_employee`

Binary/contextual variables were not log-transformed even when their
skewness values were high, because their skewness is caused by an imbalanced
binary distribution rather than a continuous financial distribution.

`profit_margin` and `management_score_proxy` are retained without log
transformation.

In [13]:
# Separate categorical, log-transformed, and regular numerical features

categorical_features = [
    "city",
    "sector"
]

log_features = [
    "monthly_revenue_proxy_inr",
    "monthly_profit_proxy_inr",
    "revenue_per_employee"
]

numerical_features = [
    feature for feature in context_features
    if feature not in categorical_features
]

regular_numeric_features = [
    feature for feature in numerical_features
    if feature not in log_features
]

print("Categorical features:", len(categorical_features))
print("Log-transformed numerical features:", len(log_features))
print("Regular numerical features:", len(regular_numeric_features))
print("Total selected features:", len(context_features))

Categorical features: 2
Log-transformed numerical features: 3
Regular numerical features: 24
Total selected features: 29


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

# Pipeline for skewed numerical features
log_pipeline = Pipeline([
    ("log", FunctionTransformer(
        np.log1p,
        feature_names_out="one-to-one"
    )),
    ("scaler", StandardScaler())
])

# Pipeline for regular numerical features
numeric_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

# Pipeline for categorical features
categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Combine all preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("log_numeric", log_pipeline, log_features),
        ("numeric", numeric_pipeline, regular_numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [16]:
# Select the final context features
X_train = train_df[context_features]
X_test = test_df[context_features]

# Fit preprocessing only on training data
X_train_transformed = preprocessor.fit_transform(X_train)

# Transform test data using the fitted training preprocessor
X_test_transformed = preprocessor.transform(X_test)

print("Train samples:", X_train_transformed.shape[0])
print("Test samples:", X_test_transformed.shape[0])
print("Final feature count:", X_train_transformed.shape[1])

Train samples: 798
Test samples: 200
Final feature count: 38


In [17]:
# Get the names of the final transformed features

feature_names = preprocessor.get_feature_names_out()

# Create DataFrames for the transformed context vectors
train_context_vectors = pd.DataFrame(
    X_train_transformed,
    columns=feature_names
)

test_context_vectors = pd.DataFrame(
    X_test_transformed,
    columns=feature_names
)

print("Train context vectors shape:", train_context_vectors.shape)
print("Test context vectors shape:", test_context_vectors.shape)

print("\nFirst 5 transformed features:")
print(feature_names[:5])

Train context vectors shape: (798, 38)
Test context vectors shape: (200, 38)

First 5 transformed features:
['log_numeric__monthly_revenue_proxy_inr'
 'log_numeric__monthly_profit_proxy_inr'
 'log_numeric__revenue_per_employee' 'numeric__female_majority_owned'
 'numeric__migrant_owner']


In [18]:
# Final verification of transformed context vectors

print("Train shape:", train_context_vectors.shape)
print("Test shape:", test_context_vectors.shape)

print("\nMissing values in train:", train_context_vectors.isna().sum().sum())
print("Missing values in test:", test_context_vectors.isna().sum().sum())

print("\nInfinite values in train:",
      np.isinf(train_context_vectors.to_numpy()).sum())
print("Infinite values in test:",
      np.isinf(test_context_vectors.to_numpy()).sum())

print("\nAll feature values are numeric:",
      train_context_vectors.dtypes.apply(
          lambda x: np.issubdtype(x, np.number)
      ).all())

Train shape: (798, 38)
Test shape: (200, 38)

Missing values in train: 0
Missing values in test: 0

Infinite values in train: 0
Infinite values in test: 0

All feature values are numeric: True


In [19]:
import joblib

# Save transformed context vectors
train_context_vectors.to_csv(
    "train_context_vectors.csv",
    index=False
)

test_context_vectors.to_csv(
    "test_context_vectors.csv",
    index=False
)

# Save the fitted preprocessing pipeline
joblib.dump(
    preprocessor,
    "preprocessing_pipeline.pkl"
)

print("Saved successfully:")
print("- train_context_vectors.csv")
print("- test_context_vectors.csv")
print("- preprocessing_pipeline.pkl")

Saved successfully:
- train_context_vectors.csv
- test_context_vectors.csv
- preprocessing_pipeline.pkl
